# Differentially Private Synthetic Tabular Data (go/dp-tabular-data-colab)

This colab demonstrates generation of differentially private synthetic data that closely resembles adult dataset data, which we treat as private/sensitive for the purpose of this colab.

### Overview
 Given a collection of records as input, we generate a collection of records as output that shares the same schema as the input data, while also trying to preserve important statistical properties of the original data. The mechanisms we currently implement follow the **Select-Measure-Generate** paradigm.  

 0. **Discretize** numeric attributes so all attributes are categorical.
 1. **Select** a collection of queries to measure — typically low-dimensional marginals, or queries of the form "SELECT col1, col2, COUNT(*) FROM DATA GROUP BY col1, col2"
 2. **Measure** the selected queries privately using a noise-addition mechanism.
 3. **Generate** synthetic data that best explains the noisy measurements.

### References

 More information about these approaches can be found at the below resources:

 * [A simple recipe for private synthetic data generation](https://differentialprivacy.org/synth-data-1/)
 * [Winning the NIST Contest: A scalable and general approach to differentially private synthetic data](https://arxiv.org/abs/2108.04978)
 * [AIM: An Adaptive and Iterative Mechanism for Differentially Private Synthetic Data](https://arxiv.org/abs/2201.12677)
 * [Graphical-model based estimation and inference for differential privacy](https://arxiv.org/abs/1901.09136)
 * [Private-PGM GitHub](https://github.com/ryan112358/private-pgm/tree/master)

### Limitations & Guarantees

* **Privacy Guarantee**: Our mechanism consume as input the desired privacy parametes epsilon and delta.  By default, the mechanisms we implement protect privacy at the row-level, wherein each individual is assumed to contribute a single record. Handling partition-level privacy, or user-level privacy when users contribute multiple records is also possible, although not currently supported.  
* **Known Domain**: Our mechanisms assume the domain is known in advance. For categorical attributes, all possible values must be either enumerated and passed in to our mechanism (preferable) or form complete finite set of all possible values. For numerical attributes, lower and upper bounds must be supplied.
* **Small subpopulations**: In general, DP is good at preserving population-level characteristics.  DP synthetic data should only be expected to preserve statistics accurately in partitions that have contributions from a sufficient number of users (typically, 100+).  By providing a large dataset consisting of more users, you can expect the synthetic data to preserve more statistics in general.
* **Large-domain categorical attributes**: Related to the above, our mechanisms are able to handle categorical attributes with a large number of possible values (> 10000), but they do this by merging attribute-values that have low signal into an extra "combined" category. To avoid this behavior, make sure the input data has been preprocessed so the number of possible values for each category are not too large (ideally, ~100).
* **Configurable Workload**: By default, the synthetic data is intended to be *generally useful* across a wide variety of statistics, although it can also be tailored to be more accurate on specific workload queries if desired.

### Supported Mechanisms and Recommendations

We currently implement three mechanisms (and more may be added in the future).

1. **The Independent Baseline** privately measures all 1-way marginals, then samples from them independently. It runs in a few seconds, but does not preserve relationships between different attributes.
2. **MST** privately selects and measures some subset of 2-way marginals, fits a model to the noisy observations, then samples synthetic data. It typically runs in around 10 minutes.
3. **AIM** iteratively selects and measures a low-order marginal, one at a time, updates its model of the data distribution, and repeats until the privacy budget is consumed. It provides the best utility but requires the most runtime.  The runtime/utility trade-off can be configured via AIM's kwargs.



In [ ]:
#@title Install libraries
!pip install git+https://github.com/google/dpsynth.git
!pip install stdmetrics
!pip install scikit-learn
!pip install kagglehub

In [ ]:
#@title Import libraries
import pandas as pd
import dpsynth
import sdmetrics

## Download Dataset

To run this notebook with a realistic dataset, we will use the Adult Income dataset from Kaggle. You will need a Kaggle account and an API token (`kaggle.json`) to download it.

In [ ]:
# Install Kaggle API
import kagglehub

kagglehub.login()

# If you are using Colab, you can alternatively set KAGGLE_USERNAME and KAGGLE_KEY
# values in user data, and then uncomment and run the following code:
#
# from colabtools import userdata
#
# os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
# os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
#
# You use userdata to keep the Kaggle API key safe. Alternatively, you can
# hardcode the values but it is not recommended due to security risks of
# leaking the API key.


# If you're not using Colab, set the env vars as appropriate for your system.
# For example, to set the env vars on Linux you can run in terminal:
# ```
# export KAGGLE_USERNAME="your_username"
# export KAGGLE_KEY="your_key"
# ```

# Download latest version
ds_path = kagglehub.dataset_download("wenruliu/adult-income-dataset")

print("Path to dataset files:", ds_path)

# Load Data and Domain

We begin by loading and inspecting the data from a csv file, and loading the domain information from a yaml file.

* **Data**: The data may contain a mix of categorical and numerical attributes. For this example, we load the UCI adult dataset, which is a small dataset derived from Census sources.
* **Domain**: The domain specificies what values are possible for each attribute in the dataset.  For numerical attributes like age, this is the minimum and maximum possible value (or some approximation thereof).  For categorical attributees like education, this is just a list, like ['HS-grad', 'Bachelors', 'Masters', ...]).  The domain information can be supplied externally in the form of a [yaml file](adult_domain.yaml).

In [ ]:
data = pd.read_csv('dpsynth/examples/adult.csv')
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
data, test = train_test_split(data, test_size=0.3, random_state=42)
print(data.shape)
print(test.shape)

In [ ]:
attribute_domains = dpsynth.domain.from_yaml_file('dpsynth/examples/adult_domain.yaml')
# Below we look at how two columns (numerical and categorical) are represented.
# Age is a numerical attribute between 17 and 90.
print(attribute_domains['age'])
# Education is a categorical value with the given possible values.
print(attribute_domains['education'])

# Generate DP Synthetic Data

Once we have the data and domain information, generating DP synthetic data is a simple one-liner shown below.  Below we use the MST mechanism, although other base mechanisms like AIM can be used with a one-line change.

In [ ]:
# MST requires couple of minutes to run, while AIM requires about
# ~40 minutes to run. For this colab notebook, we therefore only run MST.

choose_mechanism = "mst" #@param [ "mst", "aim"]

if choose_mechanism == "mst":
  mechanism = dpsynth.discrete_mechanisms.maximum_spanning_tree_mechanism
else:
  mechanism = dpsynth.discrete_mechanisms.adaptive_iterative_mechanism

print("Started generating synthetic data, this may take a while...")
import warnings
with warnings.catch_warnings():
  warnings.simplefilter("ignore") # suppress some noisy warnings
  synthetic_data = dpsynth.generate(
      data=data,
      domains=attribute_domains,
      epsilon=1.0,
      delta=1.0 / data.shape[0]**2, # 1 / N^2 where N is the number of rows in the dataset
      mechanism=mechanism,
      seed=577215664,
  )
print("Done, synthetic data is generated!")

Now let's take a look at the generated data. The number of generated rows should match the number of rows in the training dataset only approximately because noise is added to preserve anonymity.

In [ ]:
print(f"Generated {synthetic_data.shape[0]} rows of synthetic data. Training dataset has {data.shape[0]} rows (diff is {synthetic_data.shape[0] - data.shape[0]} which is {(synthetic_data.shape[0] - data.shape[0]) / data.shape[0] * 100:.2f}%).")
synthetic_data.head()

# Inspect and evaluate the synthetic data
Now we'll check that the synthetic data preserves important statistical properties of the real data.

In [ ]:
def compare_histograms(column: str | list[str]) -> pd.DataFrame:
  return pd.DataFrame({'Synthetic': synthetic_data[column].value_counts(), 'Real': data[column].value_counts()}).fillna(0)


compare_histograms('race')

In [ ]:
compare_histograms(['workclass', 'income>50K'])

In [ ]:
def compare_groupby_mean(groupby_cols: str | list[str], agg_col: str) -> pd.DataFrame:
  return pd.DataFrame({'Synthetic': synthetic_data.groupby(groupby_cols)[agg_col].mean(), 'Real': data.groupby(groupby_cols)[agg_col].mean()})

compare_groupby_mean('sex', 'age')

**1. Downstream task utility**

The efficacy score reports the utility of synthetic data in downstream tasks and compares to train data utility. Ideally, you want the synth data utility score to be *close* to that of the real data because it indicates that the synth data can be used as replacement for the real data in the application setting.

In our setting we will check on the following problem: given the tabular data we have we will train a binary classifier model to predict whether a person has income greater than 50K or not. The expectation is that performance of classifier trained on synthetic and real data will be comparable. Performance will be evaluated via f1 score.

In [ ]:
from sdmetrics.single_table import BinaryDecisionTreeClassifier
from sklearn.metrics import f1_score

In [ ]:
target="income>50K"
# only one instance of holand and not seen in training data, sdmetrics transformer doesn't handle it sadly.
test = test[test['native-country'] != 'Holand-Netherlands']

scorer=lambda y_true, y_pred: f1_score(y_true, y_pred, pos_label=0)
score_real = BinaryDecisionTreeClassifier.compute(test_data=test, train_data=data, target=target, scorer=scorer)
score_synth = BinaryDecisionTreeClassifier.compute(test_data=test, train_data=synthetic_data, target=target, scorer=scorer)

In [ ]:
print(f"Real data (baseline): {score_real}\nSynth data: {score_synth}")

**Improvements of recall metric on downstream binary classification tasks**

Now let's evaluate how helpful would it be to add our synthetic data to the real training data to improve a binary classifier. Here, we will estimate how much better recall of the classifier would be. We are using [BinaryClassifierRecallEfficacy](https://docs.sdv.dev/sdmetrics/roi-metrics/ml-augmentation/binaryclassifierrecallefficacy) estimator which outputs:

* (best) 1.0: Augmenting the real data with synthetic data improves the ML classifier's recall by the most it possibly can (100%)

* (baseline) 0.5: Augmenting the real data with synthetic data does not change the ML classifier's recall at all

* (worst) 0.0: Augmenting the real data with synthetic data decreases the ML classifier's recall by the most it possibly can (by 100%).

In [ ]:
# @title Create metadata
# ideally, this should be created manually but lazily inferring the dtype from data
import pandas as pd
import re
from datetime import datetime
from IPython.display import clear_output


# Function to infer the column type based on its data
def infer_column_type(series):
    # Check if all values are numerical (either integer or float)
    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    # Check if all values are boolean-like
    elif pd.api.types.is_bool_dtype(series):
        return "boolean"

    # Check if the column can be converted to datetime (if not already)
    elif pd.to_datetime(series, errors='coerce').notna().all():
        return "datetime"

    # If values seem like address-like, can be categorized as 'address'
    # For simplicity, assume any column with "street", "city", or "zipcode" is an address
    elif series.apply(lambda x: bool(re.search(r"(street|city|zip|address)", str(x), re.I))).any():
        return "address"

    # Otherwise, consider it categorical (text or mixed values)
    return "categorical"

# Function to read the CSV and generate the metadata dictionary
def create_metadata_from_csv(df):
    metadata = {
        "primary_key": "",  # You could infer this based on uniqueness in the data
        "columns": {}
    }

    # Iterate through columns and infer their types
    for column_name in df.columns:
        column_data = df[column_name]

        # Infer the type of the column
        column_type = infer_column_type(column_data)

        # Initialize the metadata for the column
        column_metadata = {"sdtype": column_type}

        # Add additional information for datetime columns
        if column_type == "datetime":
            column_metadata["datetime_format"] = "%Y-%m-%d"  # default format

        # Example: Adding special rules for "address" column
        if column_type == "address":
            column_metadata["pii"] = True  # Assuming addresses contain personal information

        # Add the column metadata to the dictionary
        metadata["columns"][column_name] = column_metadata

    return metadata

# Example usage
metadata_dict = create_metadata_from_csv(data)
clear_output()

In [ ]:
metadata_dict

In [ ]:
from sdmetrics.single_table.data_augmentation import BinaryClassifierRecallEfficacy
breakdown = BinaryClassifierRecallEfficacy.compute_breakdown(
    real_training_data=data,
    synthetic_data=synthetic_data,
    real_validation_data=test,
    metadata=metadata_dict,
    prediction_column_name=target,
    minority_class_label=">50K",
    classifier='XGBoost',
    fixed_precision_value=0.9,
)
breakdown

**2. Quality Report**

Here we measure two scores:

1. [Column Shapes score](https://docs.sdv.dev/sdmetrics/data-metrics/quality/quality-report#column-shapes): it measures similarity of distributions among one-column only (1-way marginals), the overall score is average over all columns. Follow the link to learn more.
1. [Column Pair score](https://docs.sdv.dev/sdmetrics/data-metrics/quality/quality-report#column-pair-trends): it measures similarity of 2d distributions among all pairs of columns (2-way marginals), the overall score is average over all pairs. Follow the link to learn more.

The scores are in [0; 1] range, the higher the better.



In [ ]:
from sdmetrics.reports.single_table import QualityReport

In [ ]:
quality_report = QualityReport()
# between real train and synthetic
_ = quality_report.generate(data, synthetic_data, metadata_dict)

Let's see Column Shapes scores per each column

In [ ]:
fig = quality_report.get_visualization(property_name='Column Shapes')
fig.show()

Let's see the scores for each pair of columns - the greener the plot is the better.

Additionally there will be "Numerical correlations" plots that show correlations inside the dataset, it is the best if two plots match as much as possible.

In [ ]:
quality_report.get_visualization(property_name='Column Pair Trends')

**3. Diagnostic Report**

Here we measure two scores:

1. [Data Validity score](https://docs.sdv.dev/sdmetrics/data-metrics/diagnostic/diagnostic-report#data-validity): checks that all values adhere to min/max boundaries if numerical and to categories sets if categorical. In range [0; 1], averaged over the columns, 1.0 is the best.
1. [Data Structure score](https://docs.sdv.dev/sdmetrics/data-metrics/diagnostic/diagnostic-report#data-structure): checks that synthetic table has the same columns as the real table. It has [0; 1] range where 1.0 is the best.

The scores have to be always 100% because they are basic validity checks.
In our case data validity score is not 100% but very close to it because of the problem with `Holand-Netherlands`.

In [ ]:
from sdmetrics.reports.single_table import DiagnosticReport

diagnostic_report = DiagnosticReport()

In [ ]:
# between real train and synthetic data
diagnostic_report.generate(data, synthetic_data, metadata_dict)

In [ ]:
fig = diagnostic_report.get_visualization(property_name='Data Validity')
fig.show()

**4. New Row Synthesis**

[This metric](https://docs.sdv.dev/sdmetrics/data-metrics/metrics-in-beta/newrowsynthesis) checks if each row in the synthetic data is unique. It has range [0; 1] and equals to $1 - \frac{MatchingSyntheticRows}{TotalSyntheticRows}$. It doesn't have to be necessarily 1.0, actually having some rows with full match might be beneficial as the attack that assumes that all rows present in the synthetic data cannot exist in the original data won't work.

(best) 1.0: The rows in the synthetic data are all new. There are no matches with the real data.

(worst) 0.0: All the rows in the synthetic data are copies of rows in the real data. Definitely undesired.

In [ ]:
# calculate whether the synthetic data is new or whether it's an exact copy of the real data

# Quite slow so, might want to consider subsampling.
from sdmetrics.single_table import NewRowSynthesis

NewRowSynthesis.compute(
    data,
    synthetic_data,
    metadata_dict
)

**5. "Privacy": Disclosure protection**

This metric checks if synthetic data provide strong disclosure protection.

(best) 1.0: The synthetic data provides strong disclosure protection. Sharing the synthetic data provides no more risk than sharing completely random values.

(worst) 0.0: The synthetic data does not provide disclosure protection. Sharing the synthetic data divulges patterns that make it easy to guess sensitive attributes.

Scores between 0.0 and 1.0 indicate the relative risk of disclosure. For example, a score of 0.825 indicates that the synthetic data has 82.5% of the protection that random data would provide.

Read more [in this documentation](https://docs.sdv.dev/sdmetrics/metrics/metrics-glossary/disclosureprotection).

In [ ]:
data.columns

In [ ]:
from sdmetrics.single_table import DisclosureProtectionEstimate

# adversary knows the age and native-country
# wants to guess sensitive column name `income`
discrete_columns = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'native-country',
    'income>50K'
]
score = DisclosureProtectionEstimate.compute_breakdown(
    real_data=data,
    synthetic_data=synthetic_data,
    known_column_names=['age', 'native-country'],
    sensitive_column_names=['income>50K'],
    continuous_column_names=list(set(data.columns)-set(discrete_columns)),
    num_rows_subsample=2500,
    num_iterations=100,
    verbose=True
)

score